# Product Recommendation Agent — Learning Notebook

This notebook walks through the building blocks of a multi-agent product recommendation system:
1. **Pydantic basics** — structured data validation
2. **Web search** — DuckDuckGo and Tavily for gathering context
3. **Agent construction** — prompt + LLM + structured output
4. **Technical specification extraction** — turning raw search text into structured specs

Run the cells top to bottom. Each section explains what the code does and why it is needed.


# Pydantic Introduction

### Check if Pydantic is Installed

`!pip show pydantic` checks whether the package is already installed and prints its version. If nothing is returned, run the next install cell.


In [9]:
!pip show pydantic

## Section 1: Pydantic Basics

Pydantic is a data-validation library. Before using it for complex agent outputs, we verify it is installed and then practice defining models.


In [10]:
!pip install --no-cache-dir pydantic

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 3.0 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.7 MB/s  0:00:00

   ---------------------------------------- 0/2 [pydantic-core]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- ------------------- 1/2 [pydantic]
   -------------------- --------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.14.6 requires pydantic<2.13,>=2.11.9, but you have pydantic 2.13.4 which is incompatible.
crewai-cli 1.14.6 requires pydantic<2.13,>=2.11.9, but you have pydantic 2.13.4 which is incompatible.
crewai-core 1.14.6 requires pydantic<2.13,>=2.11.9, but you have pydantic 2.13.4 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Install Pydantic

If `pip show` reported that pydantic was missing, this cell installs it with `--no-cache-dir` to ensure a clean download.


In [11]:
!pip show pydantic
!pip show pydantic-core

Name: pydantic
Version: 2.13.4
Summary: Data validation using Python type hints
Home-page: https://github.com/pydantic/pydantic
Author: 
Author-email: Samuel Colvin <s@muelcolvin.com>, Eric Jolibois <em.jolibois@gmail.com>, Hasan Ramezani <hasan.r67@gmail.com>, Adrian Garcia Badaracco <1755071+adriangb@users.noreply.github.com>, Terrence Dorsey <terry@pydantic.dev>, David Montague <david@pydantic.dev>, Serge Matveenko <lig@countzero.co>, Marcelo Trylesinski <marcelotryle@gmail.com>, Sydney Runkle <sydneymarierunkle@gmail.com>, David Hewitt <mail@davidhewitt.dev>, Alex Hall <alex.mojaki@gmail.com>, Victorien Plot <contact@vctrn.dev>
License-Expression: MIT
Location: C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages
Requires: annotated-types, pydantic-core, typing-extensions, typing-inspection
Required-by: chromadb, crewai, crewai-cli, crewai-core, groq, instructor, lance-namespace-urllib3-cl

### Verify Pydantic and Pydantic-Core Versions

Pydantic v2 depends on a separate Rust-based `pydantic-core` package. This cell confirms both are present.


In [18]:
from pydantic import BaseModel, Field

### Import Pydantic Classes

`BaseModel` is the parent class for every schema. `Field` adds descriptions and validation rules to individual attributes.


In [13]:
class transcript(BaseModel):
    headline : str
    content : str
    conclusion : str

### Define Your First Pydantic Model

`transcript` has three required string fields: `headline`, `content`, and `conclusion`. The type annotations (`: str`) tell Pydantic to expect strings and reject other types.


In [20]:
tr = transcript(headline = "This is introduction on pydantic", content = "Pydantic is a data validation library in Python that uses type annotations to validate and parse data. It provides a way to define data models with strict type checking, making it easier to ensure that the data being processed is in the expected format.", conclusion = "Pydantic is a powerful tool for data validation and parsing in Python, providing a way to define data models with strict type checking.")

### Create a Model Instance

We create a `transcript` object by passing values for all three fields. Pydantic validates that every field is a string before the object is created.


In [22]:
tr.conclusion

'Pydantic is a powerful tool for data validation and parsing in Python, providing a way to define data models with strict type checking.'

### Access a Field Value

Dot access (`tr.conclusion`) returns the value stored in that field. This is how downstream agents read structured LLM outputs.


In [19]:
class transcript2(BaseModel):
    headline : str = Field(description="5-10 word headline for the transcript")
    content : str
    conclusion : str

### Add Field Descriptions

`Field(description=...)` does not change validation, but it is crucial when the model is used with an LLM. The description becomes part of the prompt, guiding the model on what value to generate.


In [24]:
tr2 = transcript2(headline = "This is introduction on pydantic", content = "Pydantic is a data validation library in Python that uses type annotations to validate and parse data. It provides a way to define data models with strict type checking, making it easier to ensure that the data being processed is in the expected format.", conclusion = "Pydantic is a powerful tool for data validation and parsing in Python, providing a way to define data models with strict type checking.")

### Instantiate the Enhanced Model

The `transcript2` schema now carries field descriptions. We create an instance the same way as before.


In [25]:
tr2.headline

'This is introduction on pydantic'

### Read the Headline Field

Demonstrates that described fields are accessed the same way as plain fields.


# Product Recommendation System

## What is Product Recommendation?

A Product Recommendation System is an intelligent application that assists users in making informed purchasing decisions by automatically researching products, comparing specifications, analyzing customer reviews, evaluating alternatives, and providing personalized recommendations based on predefined criteria.

Instead of manually visiting multiple websites and comparing products, the system automates the entire decision-making process and delivers a concise recommendation with supporting evidence.

---

## Product Recommendation Workflow

### Phase 1: Discovery Phase

**Objective:** Identify relevant products and brands based on user requirements.

#### Activities

- Understand user intent
- Extract user requirements
  - Product category
  - Budget
  - Brand preference
  - Use case
- Discover available products
- Identify top brands and models

#### Example

```text
User Query:
"Recommend a laptop under ₹1,50,000 for AI development"

Output:
- ASUS ROG Strix
- Lenovo Legion
- HP Omen
- Acer Predator
```

---

### Phase 2: Specification Research

**Objective:** Collect and compare technical specifications.

#### Activities

- Gather product specifications
- Compare technical features
- Identify strengths and weaknesses
- Verify compatibility requirements
- Build a specification comparison matrix

#### Example

```text
Specification Comparison

Product A
- RTX 5070
- 32 GB RAM
- 1 TB SSD

Product B
- RTX 5060
- 16 GB RAM
- 512 GB SSD
```

---

### Phase 3: Review Analysis

**Objective:** Analyze customer feedback and expert reviews.

#### Activities

- Collect customer reviews
- Analyze review sentiment
- Identify recurring issues
- Identify commonly praised features
- Evaluate overall user satisfaction
- Analyze expert recommendations

#### Example

```text
Positive Findings
- Excellent battery life
- Strong performance

Negative Findings
- Heating issues
- Loud fan noise
```

---

### Phase 4: Advisory Verdict

**Objective:** Provide a final recommendation.

#### Activities

- Rank products
- Justify recommendations
- Highlight trade-offs
- Provide final purchasing advice
- Explain recommendation rationale

#### Example

```text
Recommended Product:
ASUS ROG Strix G16

Reason:
- Best GPU in budget
- Future-proof for AI workloads
- Better thermals compared to competitors
```

---

## End-to-End Process Flow

```text
User Query
      │
      ▼
Requirement Extraction
      │
      ▼
Product Discovery
      │
      ▼
Specification Collection
      │
      ▼
Review Analysis
      │
      ▼
Product Comparison
      │
      ▼
Scoring & Ranking
      │
      ▼
Recommendation Generation
      │
      ▼
Final Advisory Report
```

---

# Functional Requirements

## User Input

The system shall accept:

- Product category
- Budget range
- Preferred brands
- Required specifications
- Intended use case
- Additional constraints

---

## Product Discovery

The system shall:

- Search products from trusted sources
- Identify top products within the requested category
- Filter products based on user-defined criteria
- Remove duplicate products

---

## Specification Analysis

The system shall:

- Extract product specifications
- Normalize technical attributes
- Compare products across common metrics
- Highlight differences and advantages

---

## Review Analysis

The system shall:

- Analyze customer reviews
- Identify positive sentiments
- Identify negative sentiments
- Detect recurring complaints
- Generate review summaries

---

## Recommendation Engine

The system shall:

- Score products
- Rank alternatives
- Generate recommendations
- Explain recommendation reasoning
- Provide confidence scores

---

# Non-Functional Requirements

## Performance

- Response time should be less than 30 seconds
- Support concurrent user requests

## Reliability

- Handle source failures gracefully
- Provide fallback recommendations when data is unavailable

## Scalability

- Support multiple product categories
- Handle large product datasets

## Explainability

- Every recommendation must include supporting evidence
- Users must understand why a product was selected

---

# Guardrails

The system must operate only within predefined boundaries.

## Product Category Guardrails

### Allowed Categories

```text
Electronics
Laptops
Mobile Phones
Monitors
Headphones
Home Appliances
Books
```

### Restricted Categories

```text
Weapons
Illegal Products
Restricted Substances
Counterfeit Products
```

---

## Budget Guardrails

- Validate budget inputs
- Reject unrealistic budgets
- Ensure minimum and maximum budget thresholds

### Example

```text
Laptop under ₹500
→ Invalid Request
```

---

## Data Source Guardrails

### Approved Sources

```text
Amazon
Best Buy
Newegg
Manufacturer Websites
Official Brand Stores
```

### Restricted Sources

```text
Unknown Websites
Unverified Review Platforms
Suspicious Data Sources
```

---

## Recommendation Guardrails

The system shall:

- Avoid biased recommendations
- Explain ranking logic
- Disclose uncertainty when confidence is low
- Prevent hallucinated specifications
- Provide evidence-based recommendations only

---

# Agentic AI Requirement

## Constraint

The solution must automate the complete workflow **without using orchestration frameworks**, including but not limited to:

- LangGraph
- CrewAI
- AutoGen
- Semantic Kernel

---

## Implementation Approach

The workflow shall be orchestrated using native Python classes, functions, and custom state management.

```text
## End-to-End Agent Workflow

```text
User Query
      │
      ▼
Agent 0: Discovery Agent
      │
      ├── Tool: Product Discovery Tool
      │
      ├── Searches the internet for relevant products
      │
      ├── Raw search results may contain:
      │     - Relevant products
      │     - Irrelevant products
      │     - Missing information
      │     - Inconsistent formatting
      │
      ├── LLM analyzes search results
      │
      ├── Problem:
      │     LLM output is non-deterministic and may not
      │     follow the same format every time.
      │
      ├── Solution:
      │     Use Pydantic models to define a strict output schema.
      │
      ├── Structured Output:
      │     LangChain's `with_structured_output()`
      │     method is used to bind the Pydantic schema
      │     to the LLM response.
      │
      ├── Benefits:
      │     - Type safety
      │     - Output validation
      │     - Consistent response structure
      │     - Easier downstream processing
      │
      └── Output:
            List of discovered products

            Example:

            [
              {
                "brand": "ASUS",
                "model": "ROG Strix G16",
                "reason": "Popular AI development laptop"
              },
              {
                "brand": "Lenovo",
                "model": "Legion Pro 5",
                "reason": "High GPU performance"
              }
            ]

      │
      ▼
Agent 1: Specification Agent
      │
      ├── Input:
      │     List of discovered products
      │
      ├── Tool:
      │     Product Specification Tool
      │
      ├── Collects:
      │     - CPU
      │     - GPU
      │     - RAM
      │     - Storage
      │     - Display
      │     - Battery
      │
      ├── Structured Output:
      │     ProductSpecification Pydantic Model
      │
      └── Output:
            Normalized product specifications

      │
      ▼
Agent 2: Review Analysis Agent
      │
      ├── Input:
      │     Product specifications
      │
      ├── Tools:
      │     - Reddit Search
      │     - Review Websites
      │     - YouTube Reviews
      │
      ├── Performs:
      │     - Sentiment Analysis
      │     - Pros Identification
      │     - Cons Identification
      │     - Common Complaints Detection
      │
      ├── Structured Output:
      │     ReviewAnalysis Pydantic Model
      │
      └── Output:
            Product review insights

      │
      ▼
Agent 3: Recommendation Agent
      │
      ├── Input:
      │     - Product List
      │     - Specifications
      │     - Review Analysis
      │
      ├── Performs:
      │     - Product Scoring
      │     - Product Ranking
      │     - Trade-off Analysis
      │     - Confidence Assessment
      │
      ├── Structured Output:
      │     RecommendationReport Pydantic Model
      │
      └── Output:
            Ranked recommendations

      │
      ▼
Final Advisory Report
      │
      ├── Top Recommended Product
      ├── Alternative Options
      ├── Specification Comparison
      ├── Review Summary
      ├── Pros & Cons
      ├── Recommendation Reasoning
      └── Confidence Score
```

---

## Why Structured Output is Important

Without Structured Output:

```python
"ASUS ROG Strix is a great laptop..."
```

or

```python
{
  "laptop": "ASUS ROG Strix"
}
```

or

```python
[
  "ASUS",
  "Lenovo"
]
```

The format may change every time.

With Structured Output + Pydantic:

```python
class Product(BaseModel):
    brand: str
    model: str
    reason: str
```

The LLM is forced to return:

```json
{
  "brand": "ASUS",
  "model": "ROG Strix G16",
  "reason": "Excellent GPU performance for AI workloads"
}
```

This makes the output reliable and allows the next agent to consume the data without additional parsing or validation logic.
```

### Agent Responsibilities

#### Discovery Agent

- Understand user requirements
- Identify products and brands
- Build candidate product list

#### Specification Agent

- Collect specifications
- Normalize attributes
- Generate comparison matrix

#### Review Analysis Agent

- Gather reviews
- Perform sentiment analysis
- Identify common strengths and weaknesses

#### Recommendation Agent

- Score products
- Rank alternatives
- Generate recommendation rationale
- Produce final advisory report

---

# Expected Output

The system shall generate:

- Product Summary
- Specification Comparison
- Review Insights
- Pros and Cons
- Product Ranking
- Final Recommendation
- Recommendation Justification
- Confidence Score

## Section 2: Product Recommendation System Overview

This section introduces the four-phase workflow of the recommendation system, then installs the libraries needed to build the agents.


### Install Project Dependencies

Installs LangChain, OpenAI integration, DuckDuckGo search, and Pydantic. These are the core libraries for building the agents.


In [26]:
!pip install langchain langchain-openai langchain-community duckduckgo-search pydantic

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 2.2 MB/s eta 0:00:01
   ---------------------- ----------------- 1.3/2.4 MB 2.3 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 2.3 MB/s  0:00:01
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 2.4 MB/s  0:00:00
   ---------------------------------------- 0.0/


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Search the Web with DuckDuckGo

`DDGS()` is the DuckDuckGo search client. This cell searches for "Best AI laptops 2026" and prints the first 5 text results.

**Why this matters:** Agents need fresh, real-world context. Without web search, the LLM would rely only on its training data, which can be outdated.


In [31]:
## Bad context provides Bad results. So we need to provide good context to the agent. We can use duckduckgo-search to get the context for the agent.
from duckduckgo_search import DDGS

with DDGS() as ddgs:
    results = list(ddgs.text("Best AI laptops 2026", max_results=5))

print(results)

C:\Users\BhargavDharan\AppData\Local\Temp\ipykernel_19752\2974331409.py:3: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


[{'title': 'BEST Online Payment', 'href': 'https://www.bestundertaking.net/', 'body': 'Hello, Consumer Enter your 9 digit consumer number and pay online Quick Pay'}, {'title': 'The Brihanmumbai Electric Supply & Transport Undertaking (BEST) | of ...', 'href': 'https://www.bestundertaking.com/', 'body': 'Official website of Brihanmumbai Electric Supply & Transport (BEST) Undertaking – providing reliable transport and electricity services across Mumbai.'}, {'title': 'BEST Definition & Meaning - Merriam-Webster', 'href': 'https://www.merriam-webster.com/dictionary/best', 'body': '2 days ago · In the best of all possible worlds, no one would be without food and clean water. I have a wonderful family and a great job, so I feel that I have the best of both worlds.'}, {'title': 'BEST | English meaning - Cambridge Dictionary', 'href': 'https://dictionary.cambridge.org/dictionary/english/best', 'body': 'best adjective, adverb (HIGHEST QUALITY) of the highest quality, to the greatest degree, in 

### Try a More Specific Search Query

Adding keywords like "machine learning", "AI development", and "RTX GPU" focuses the search on laptops relevant to our use case.


In [32]:
ddgs.text(
    "best laptops for machine learning AI development RTX GPU 2026",
    max_results=10
)

[{'title': 'BEST Online Payment',
  'href': 'https://www.bestundertaking.net/',
  'body': 'Hello, Consumer Enter your 9 digit consumer number and pay online Quick Pay'},
 {'title': 'The Brihanmumbai Electric Supply & Transport Undertaking (BEST) | of ...',
  'href': 'https://www.bestundertaking.com/',
  'body': 'Official website of Brihanmumbai Electric Supply & Transport (BEST) Undertaking – providing reliable transport and electricity services across Mumbai.'},
 {'title': 'BEST Definition & Meaning - Merriam-Webster',
  'href': 'https://www.merriam-webster.com/dictionary/best',
  'body': '2 days ago · In the best of all possible worlds, no one would be without food and clean water. I have a wonderful family and a great job, so I feel that I have the best of both worlds.'},
 {'title': 'BEST | English meaning - Cambridge Dictionary',
  'href': 'https://dictionary.cambridge.org/dictionary/english/best',
  'body': 'best adjective, adverb (HIGHEST QUALITY) of the highest quality, to the g

### Search Within a Specific Site

The `site:tomshardware.com` operator restricts results to a trusted review site. This improves result quality when you know a reliable source.


In [33]:
results = list(
    ddgs.text(
        "best laptops for AI development site:tomshardware.com",
        max_results=5
    )
)

### Build a Reusable Search Function

`search_products(query)` wraps the DuckDuckGo client so other cells can call it with a simple string. It returns a list of result dictionaries.


In [34]:
from duckduckgo_search import DDGS

def search_products(query):
    with DDGS() as ddgs:
        results = list(
            ddgs.text(
                query,
                max_results=10
            )
        )

    return results

results = search_products(
    "best laptops for AI development RTX GPU 2026"
)

for r in results:
    print(r["title"])

C:\Users\BhargavDharan\AppData\Local\Temp\ipykernel_19752\3542948474.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


BEST Online Payment
The Brihanmumbai Electric Supply & Transport Undertaking (BEST) …
BEST Definition & Meaning - Merriam-Webster
BEST | English meaning - Cambridge Dictionary
BEST MOBILE PHONES - Gadgets 360


### Load API Keys from `.env`

`load_dotenv()` reads environment variables from a `.env` file. We print `TAVILY_API_KEY` to confirm it was loaded (the actual value is hidden in normal use).


In [27]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("TAVILY_API_KEY"))

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


### Install Tavily Search Package

Tavily is an AI-native search API. This cell installs its Python client so we can compare Tavily results with DuckDuckGo results.


In [28]:
!pip install tavily-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Search with Tavily

Create a `TavilyClient` using the API key and search for the best AI laptops. Tavily returns structured JSON with title, URL, content, and relevance score.


In [29]:
from tavily import TavilyClient
import os

client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)

result = client.search(
    query="Best laptops for AI development 2026",
    max_results=5
)

print(result)

{'query': 'Best laptops for AI development 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.creativebloq.com/tech/laptops/the-best-ai-laptops-future-proof-your-creative-work-with-an-onboard-npu', 'title': 'The best AI laptops: future-proof your creative work with an onboard ...', 'content': "The Asus Zenbook 14 OLED (UX3405) is the AI laptop we'd recommend to most creatives in 2026. It strikes the best balance of performance", 'score': 0.92652357, 'raw_content': None}, {'url': 'https://www.tomsguide.com/best-picks/best-ai-laptop', 'title': "Best AI laptops in 2026, tested by experts | Tom's Guide", 'content': "Join the Tom's Guide Club for quick access. Enter your email below and we'll send confirmation, and sign you up to our newsletter. Here at Tom’s Guide our expert editors are committed to bringing you the best news, reviews and guides to help you stay informed and ahead of the curve! Sign up to get the latest updates on all of your

### Pretty-Print Tavily Results

Loop through the Tavily result list and print each item's title, URL, and content in a readable format.


In [30]:
for item in result["results"]:
    print("=" * 80)
    print("Title :", item["title"])
    print("URL   :", item["url"])
    print("Content:")
    print(item["content"])
    print()

Title : The best AI laptops: future-proof your creative work with an onboard ...
URL   : https://www.creativebloq.com/tech/laptops/the-best-ai-laptops-future-proof-your-creative-work-with-an-onboard-npu
Content:
The Asus Zenbook 14 OLED (UX3405) is the AI laptop we'd recommend to most creatives in 2026. It strikes the best balance of performance

Title : Best AI laptops in 2026, tested by experts | Tom's Guide
URL   : https://www.tomsguide.com/best-picks/best-ai-laptop
Content:
Join the Tom's Guide Club for quick access. Enter your email below and we'll send confirmation, and sign you up to our newsletter. Here at Tom’s Guide our expert editors are committed to bringing you the best news, reviews and guides to help you stay informed and ahead of the curve! Sign up to get the latest updates on all of your favorite content! Tom&#039;s AI Guide. With coverage on everything from exciting product launches to essential software updates, this is your go-to source for the latest updates on all

### Update DuckDuckGo Search Package

The `-U` flag upgrades `ddgs` to the latest version, which can fix compatibility issues with LangChain's wrapper.


In [81]:
!pip install -U ddgs

Defaulting to user installation because normal site-packages is not writeable
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl (369 kB)

   ----- ---------------------------------- 1/7 [socksio]
   ----- ---------------------------------- 1/7 [socksio]
   ----------- ---------------------------- 2/7 [hyperframe]
   ----------- ---------------------------- 2/7 [hyperframe]
   ----------------- ---------------------- 3/7 [hpack]
   ----------------- ---------------------- 3/7 [hpack]
   ---------------------- ----------------- 4/7 [fake-useragent]
   ---------------------- ----------------- 4/7 [fake-useragent]
   ---------------------------- ----------- 5/7 [h2]
   ---------------------------- ----------- 5/7 [h2]
   ---------------------------- ----------- 5/7 [h2]
   ---------------------------- ----------- 5/7 [h2]
   ---------------------------------- ----- 6/7 [ddgs]
   ---------------------------------- ---


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\utils\_process_win32.py:138: Resourc

### Import LangChain Agent Components

Key imports:
- `ChatOpenAI` — the LLM
- `BaseTool`, `tool` — decorator for making functions usable as tools
- `ChatPromptTemplate` — prompt templates
- `DuckDuckGoSearchResults`, `DuckDuckGoSearchRun` — LangChain search wrappers


In [69]:
import os
from datetime import date, timedelta
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI

from langchain_core.tools import BaseTool, tool


from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

from langchain_community.tools import DuckDuckGoSearchResults, DuckDuckGoSearchRun

### Load Environment Variables

Run `load_dotenv(override=True)` after imports to make sure all API keys are available before creating clients.


In [17]:
load_dotenv(override=True)

True

### Initialize the LLM

`ChatOpenAI(model="gpt-5", temperature=0.5)` creates the language model. If `gpt-5` is unavailable, switch to `gpt-4o` or `gpt-4o-mini`.


In [22]:
llm = ChatOpenAI(model="gpt-5", temperature=0.5)

### Check Today's Date

`date.today()` returns the current date. We will append the year to search queries so agents look at current models.


In [67]:
date.today()

datetime.date(2026, 5, 31)

### Define a Product Discovery Tool

The `@tool` decorator converts `discover_product_tool(category)` into a LangChain tool. It uses `DuckDuckGoSearchResults` to fetch product overview articles for the given category in India.

**Note:** The function catches exceptions and returns an empty string on failure so the notebook keeps running.


In [3]:
# # @tool

# def discover_product_tool(category: str) -> str:
#     """
#     This tool uses the DuckDuckGoSearchResults tool to search for the best products in the given category.
#     It returns a list of product names.
#     """
#     search_results = DuckDuckGoSearchResults().run(f"best {category} 2026")
#     today = date.today()
#     country = "India"
#     try:
#         return search_results.run(f"best {category} current models overview articles as of {today} in {country}", max_results=10)
#     except Exception as e:
#         print(f"Error occurred while searching: {str(e)}")
#         return []
    


def discover_product_tool(category: str) -> str:
    """
    Search for products in a given category.

    Args:
        category (str): Product category (e.g., laptops, mobiles)

    Returns:
        str: Search results or error message
    """

    try:
        today = date.today()
        country = "India"

        search_tool = DuckDuckGoSearchResults()

        search_results = search_tool.run(
            f"best {category} current models overview articles as of {today} in {country}"
        )

        return search_results

    except Exception as e:
        print(f"Error occurred while searching: {e}")
        return ""

### Test the Discovery Tool

Call the tool with the category "laptops" and print the raw search string it returns. This is the context that will later feed the discovery agent.


In [4]:
result = discover_product_tool("laptops")

print(result)

snippet: Best Laptop 2026: Portable computers for work, gaming and more., title: trustedreviews.com/best/best-laptops-3431966, link: https://www.trustedreviews.com/best/best-laptops-3431966, snippet: Ranking of the best laptop CPUs based on real-world tests, benchmarks, power efficiency, iGPU performance and other factors., title: Laptop Processors Ranking List [2026] - NanoReview, link: https://nanoreview.net/en/cpu-list/laptop-chips-rating, snippet: Plastindia Description..., title: Plastindia, link: https://plastindia.org/plastindia-2026, snippet: Support for Spatial Audio when playing music or video with Dolby Atmos on built-in speakers Spatial Audio with dynamic head tracking when using supported models of AirPods, AirPods Pro, and AirPods Max 14., title: Compare Mac Models - Apple, link: https://www.apple.com/mac/compare/


### Inspect the Tool Object

Printing `discover_product_tool` shows its LangChain tool metadata, such as name, description, and expected arguments.


In [5]:
discover_product_tool

<function __main__.discover_product_tool(category: str) -> str>

### Common Mistake: Tool vs. Tool Result

Assigning `result = discover_product_tool` stores the function itself, not its output. The cell prints the function object to illustrate why you must call it with parentheses.


In [89]:
result = discover_product_tool

print(result)

<function discover_product_tool at 0x0000020D1FC19EE0>


### Define a Discovery Output Schema

`ProductDiscoveryAgent` tells the LLM what to return: a confirmed category and a list of product names. `Field(...)` descriptions guide the LLM's extraction.


In [6]:
class ProductDiscoveryAgent(BaseModel):
    # llm: ChatOpenAI
    # tools: List[BaseTool]
    category_confirmed : str = Field(description="The category for which the products are to be discovered")
    product_name : List[str] = Field(description="List of product names discovered for the category")
    

NameError: name 'BaseModel' is not defined

### Build a Discovery Prompt

`ChatPromptTemplate.from_messages([...])` defines the conversation structure. We use a system message plus a human message that contains `{user_query}` and `{search_context}` placeholders.


In [1]:
discovery_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        # ("human", "Discover the best products in the category of {category_confirmed} and provide a list of product names.")
        ("human", "User Criteria: {user_query}. Search findings: {search_context}")
    ]
)

discovery_prompt


NameError: name 'ChatPromptTemplate' is not defined

### Create the Structured Discovery Chain

`discovery_prompt | llm.with_structured_output(ProductDiscoveryAgent)` connects the prompt to the LLM and binds the output schema. The result of running this chain will be a `ProductDiscoveryAgent` object.


In [102]:
structured_discover = discovery_prompt | llm.with_structured_output(ProductDiscoveryAgent)

NameError: name 'ProductDiscoveryAgent' is not defined

### Set the User Query

Store the recommendation request in a variable so it can be reused across multiple agent runs.


In [100]:
user_query = "Best laptops for AI development under 150000 INR"

### Run the Discovery Chain (Correct Invoke Syntax)

**Important:** `invoke()` expects a single dictionary of inputs, not keyword arguments.

This cell passes the user query and the output of `discover_product_tool("laptops")` to the chain.


In [106]:
# BUG FIX: invoke() takes a DICTIONARY, not keyword arguments
# Wrong: structured_discover.invoke(user_query=..., search_context=...)
# Right: structured_discover.invoke({"user_query": ..., "search_context": ...})
discovery_result = structured_discover.invoke(
    {
        "user_query": user_query,
        "search_context": discover_product_tool("laptops")
    }
)

print(discovery_result)
print(discovery_result.model_dump())

NameError: name 'structured_discover' is not defined

### Re-import for a Cleaner Schema

We redefine `ProductDiscoveryAgent` with more concise field descriptions and rebuild the discovery prompt with a clearer system role.


In [7]:
from pydantic import BaseModel, Field
from typing import List

from langchain_core.prompts import ChatPromptTemplate

C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Rebuild the Discovery Schema

A cleaner `ProductDiscoveryAgent` with `category_confirmed` and `product_name` fields and clearer descriptions.


In [8]:
class ProductDiscoveryAgent(BaseModel):

    category_confirmed: str = Field(
        description="Detected category"
    )

    product_name: List[str] = Field(
        description="List of products"
    )

### Rebuild the Discovery Prompt

A better system prompt: "You are a product discovery expert." The human message now clearly separates the user query and the search results.


In [9]:
discovery_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a product discovery expert."),
        (
            "human",
            """
            User Query:
            {user_query}

            Search Results:
            {search_context}
            """
        )
    ]
)

### Verify Required Objects Exist

Before rebuilding the chain, confirm that `llm`, `discovery_prompt`, and `ProductDiscoveryAgent` are defined in memory. This helps debug "name not defined" errors.


In [19]:
print("llm exists:", "llm" in globals())
print("prompt exists:", "discovery_prompt" in globals())
print("schema exists:", "ProductDiscoveryAgent" in globals())

llm exists: True
prompt exists: True
schema exists: True


### Rebuild the Structured Discovery Chain

Combine the new prompt and schema into a fresh chain. This overwrites the earlier `structured_discover` variable.


In [43]:
structured_discover = (
    discovery_prompt
    | llm.with_structured_output(ProductDiscoveryAgent)
)

### Run the Improved Discovery Chain

Invoke the rebuilt chain with the same user query and search context. The output should be a cleaner `ProductDiscoveryAgent` object.


In [24]:
user_query = "Best laptops for AI development under 150000 INR"

discovery_result = structured_discover.invoke(
    {
        "user_query": user_query,
        "search_context": discover_product_tool("laptops")
    }
)

### Print the Discovery Result

Print both the Pydantic object and its dictionary representation to compare the two formats.


In [25]:
print(discovery_result)
print(discovery_result.model_dump())

category_confirmed='Laptops for AI development under ₹1,50,000 (India)' product_name=['Acer Predator Helios Neo 16 (RTX 4060, 140W)', 'Lenovo Legion 5 (2023/2024, RTX 4060)', 'ASUS TUF Gaming A15 FA507 (RTX 4070, 140W)', 'HP Omen 16 (RTX 4060)', 'ASUS Vivobook Pro 15/16 OLED (RTX 4060)', 'Dell G15 5530/5535 (RTX 4060)', 'Lenovo LOQ 15/16 (RTX 4060)', 'MSI Katana 15 (RTX 4060)', 'Acer Nitro 16 (RTX 4060)', 'ASUS ROG Strix G16 (RTX 4060)', 'Gigabyte G5/G6 (RTX 4060)', 'Apple MacBook Air 15 (M3, 16GB) – for cloud‑first AI dev']
{'category_confirmed': 'Laptops for AI development under ₹1,50,000 (India)', 'product_name': ['Acer Predator Helios Neo 16 (RTX 4060, 140W)', 'Lenovo Legion 5 (2023/2024, RTX 4060)', 'ASUS TUF Gaming A15 FA507 (RTX 4070, 140W)', 'HP Omen 16 (RTX 4060)', 'ASUS Vivobook Pro 15/16 OLED (RTX 4060)', 'Dell G15 5530/5535 (RTX 4060)', 'Lenovo LOQ 15/16 (RTX 4060)', 'MSI Katana 15 (RTX 4060)', 'Acer Nitro 16 (RTX 4060)', 'ASUS ROG Strix G16 (RTX 4060)', 'Gigabyte G5/G6 (RT

### Display the Result Object

Jupyter automatically renders the last expression in a cell. This cell shows the default string representation of the Pydantic object.


In [27]:
discovery_result

ProductDiscoveryAgent(category_confirmed='Laptops for AI development under ₹1,50,000 (India)', product_name=['Acer Predator Helios Neo 16 (RTX 4060, 140W)', 'Lenovo Legion 5 (2023/2024, RTX 4060)', 'ASUS TUF Gaming A15 FA507 (RTX 4070, 140W)', 'HP Omen 16 (RTX 4060)', 'ASUS Vivobook Pro 15/16 OLED (RTX 4060)', 'Dell G15 5530/5535 (RTX 4060)', 'Lenovo LOQ 15/16 (RTX 4060)', 'MSI Katana 15 (RTX 4060)', 'Acer Nitro 16 (RTX 4060)', 'ASUS ROG Strix G16 (RTX 4060)', 'Gigabyte G5/G6 (RTX 4060)', 'Apple MacBook Air 15 (M3, 16GB) – for cloud‑first AI dev'])

### Access the Detected Category

`discovery_result.category_confirmed` extracts just the category string from the structured output.


In [28]:
discovery_result.category_confirmed

'Laptops for AI development under ₹1,50,000 (India)'

### Access the Product List

`discovery_result.product_name` returns the list of discovered product names. This list becomes the input for the specification agent.


In [29]:
discovery_result.product_name

['Acer Predator Helios Neo 16 (RTX 4060, 140W)',
 'Lenovo Legion 5 (2023/2024, RTX 4060)',
 'ASUS TUF Gaming A15 FA507 (RTX 4070, 140W)',
 'HP Omen 16 (RTX 4060)',
 'ASUS Vivobook Pro 15/16 OLED (RTX 4060)',
 'Dell G15 5530/5535 (RTX 4060)',
 'Lenovo LOQ 15/16 (RTX 4060)',
 'MSI Katana 15 (RTX 4060)',
 'Acer Nitro 16 (RTX 4060)',
 'ASUS ROG Strix G16 (RTX 4060)',
 'Gigabyte G5/G6 (RTX 4060)',
 'Apple MacBook Air 15 (M3, 16GB) – for cloud‑first AI dev']

### Define a Technical Specs Research Tool

`research_technical_specs` is a `@tool`-decorated function that searches DuckDuckGo for a product's official technical specifications.

**Note:** It uses `DuckDuckGoSearchRun` (a simpler wrapper) rather than `DuckDuckGoSearchResults`.


In [55]:
@tool

def research_technical_specs(product_name: str) -> str:
    """
    Research technical specifications for a given product.

    Args:
        product_name (str): Name of the product to research

    Returns:
        str: Technical specifications for the product
    """
    
    search = DuckDuckGoSearchRun()
    
    try:
        specs = search.run(f"{product_name} official technical specifications 2026 laptops from india")
        return specs
    except Exception as e:
        return f"Could not retrieve technical specifications for {product_name}. Error: {str(e)}"
    


### Define the Technical Specification Schema

`TechnicalSpecs` requires CPU, GPU, RAM, storage, display, battery, weight, price, and an AI-focused summary. These fields mirror the final recommendation comparison table.


In [71]:
# class TechnicalSpecs(BaseModel):
#     product_name: str = Field(description="Name of the product")
#     key_specs: str = Field(description="Technical specifications of the product")
#     summary: str = Field(description="Summary of the technical specifications")

class TechnicalSpecs(BaseModel):

    product_name: str = Field(
        description="Product Name"
    )

    cpu: str = Field(
        description="Processor"
    )

    gpu: str = Field(
        description="Graphics Card"
    )

    ram: str = Field(
        description="RAM Capacity"
    )

    storage: str = Field(
        description="Storage Capacity"
    )

    display: str = Field(
        description="Display Information"
    )

    battery: str = Field(
        description="Battery Information"
    )

    weight: str = Field(
        description="Device Weight"
    )

    price: str = Field(
        description="Approximate Price"
    )

    summary: str = Field(
        description="AI-focused technical summary"
    )

### Create a Structured Specs LLM

`llm.with_structured_output(TechnicalSpecs)` tells the LLM to return output matching the `TechnicalSpecs` schema.


In [62]:
technical_specs_llm = llm.with_structured_output(TechnicalSpecs)

### Build the Specs Extraction Prompt

The prompt tells the LLM to act as a hardware research expert and extract specific fields from raw research data.


In [63]:
technical_specs_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a hardware research expert.

            Extract technical specifications from the provided research data.

            Focus on:
            - CPU
            - GPU
            - RAM
            - Storage
            - Display
            - Battery
            - Weight

            Generate a concise summary.
            """
        ),
        (
            "human",
            """
            Product Name:
            {product_name}

            Research Data:
            {research_data}
            """
        )
    ]
)

### Compose the Specs Chain

Chain the prompt and the structured LLM together. The resulting `technical_specs_chain` can be invoked with a product name and research data.


In [64]:
technical_specs_chain = (
    technical_specs_prompt
    | technical_specs_llm
)

### Run Agent 1: Collect Technical Specifications

Loop over every product name discovered earlier. For each product:
1. Call `research_technical_specs.invoke({"product_name": product})` to get raw data.
2. Call `technical_specs_chain.invoke(...)` to extract structured specs.
3. Append the result to `all_specs`.

**Bug-fix note:** `all_specs = []` must be defined before the loop, otherwise it resets each iteration.


In [74]:
# Agent 1 : Fetching Technical Specifications

# for product in discovery_result.product_name:
#     # specs = research_technical_specs(product)
#     print(f"--- processing : {product} ----")
#     print(f"Agent1 : Fetching Technical Specifications ......")
#     # spec_raw_data = research_technical_specs.invoke({"product_name" : product})
#     product = "Acer Predator Helios Neo 16"

#     research_data = research_technical_specs.invoke(
#         {"product_name": product}
#     )

#     print(research_data)
#     # print(spec_raw_data)
    
#     break


# BUG FIX: all_specs = [] was INSIDE the loop, so it reset every iteration
# and only kept the last product's specs. Moved it BEFORE the loop.
all_specs = []

for product in discovery_result.product_name:

    print(f"--- Processing: {product} ---")

    # Step 1: Search for raw spec data using DuckDuckGo
    research_data = research_technical_specs.invoke(
        {"product_name": product}
    )

    # Step 2: LLM extracts structured specs from raw data
    spec_result = technical_specs_chain.invoke(
        {
            "product_name": product,
            "research_data": research_data
        }
    )

    all_specs.append(spec_result)

print(f"\nCollected specs for {len(all_specs)} products.")

### Display the Specification Comparison

Iterate over `all_specs` and print each product's CPU, GPU, RAM, storage, display, battery, weight, price, and summary in a comparison layout.


In [ ]:
# Display all collected specs
print("=" * 70)
print("SPECIFICATION COMPARISON")
print("=" * 70)

for spec in all_specs:

    print("\n" + "-"*70)
    print(f"Product : {spec.product_name}")
    print(f"CPU     : {spec.cpu}")
    print(f"GPU     : {spec.gpu}")
    print(f"RAM     : {spec.ram}")
    print(f"Storage : {spec.storage}")
    print(f"Display : {spec.display}")
    print(f"Battery : {spec.battery}")
    print(f"Weight  : {spec.weight}")
    print(f"Price   : {spec.price}")
    print(f"\nSummary: {spec.summary}")

### Check the Tool Type

`type(research_technical_specs)` confirms the decorated function is now a LangChain `StructuredTool` object.


In [45]:
print(type(research_technical_specs))

<class 'function'>


## Section 3: Product Recommendation with Tavily Search API

This section repeats the discovery idea using Tavily instead of DuckDuckGo. Tavily returns cleaner structured results, which can improve extraction accuracy.


# Product Recommendation with search API

### Install Tavily Package

Installs the Tavily client for the second time in the notebook to ensure it is available in this section.


In [41]:
# Using Tavily Search API to get the search results for the query "Best laptops for AI development 2026" - Freemium API with 1000 free searches per month

!pip install tavily-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\utils\_process_win32.py:138: Resourc

### Import Tavily Components

Imports are similar to the DuckDuckGo section, but `TavilyClient` replaces the DuckDuckGo wrappers.


In [42]:
import os
from datetime import date, timedelta
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from tavily import TavilyClient

from langchain_openai import ChatOpenAI

from langchain_core.tools import BaseTool

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

### Initialize Tavily Client

Load environment variables and create the Tavily client using `TAVILY_API_KEY`.


In [43]:
load_dotenv()

tavily_client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)

### Test Tavily Search

Run a targeted query for laptops under ₹1,50,000 for AI development and print the raw JSON response.


In [45]:
# Product Discovery Tool - search Testing

results = tavily_client.search(
    query="Best laptops for AI development under 150000 INR",
    max_results=5
)

print(results)

{'query': 'Best laptops for AI development under 150000 INR', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.facebook.com/groups/ioeentrancepreparationnepal/posts/2556792671369841', 'title': 'Best laptop for programming with AI features under 1 lakhs?', 'content': 'Best laptop for programming with AI features under 1 lakhs ... Laptop suggestions for programming beginners under Rs 27,000. Dakshta', 'score': 0.9998523, 'raw_content': None}, {'url': 'https://www.youtube.com/watch?v=SbVU3VYxkwk', 'title': 'The Ultimate AI/ML Laptop Buying Guide in 2026! - YouTube', 'content': '... under $1000, affordable laptops for AI and deep learning ... best for AI development in 2024, best laptop for training AI models at home.', 'score': 0.99983656, 'raw_content': None}, {'url': 'https://www.youtube.com/watch?v=RE5NE-mBULk', 'title': 'Best Laptops for AI Engineer (2026) Updated! - YouTube', 'content': 'Best Buying Links 1. MSI Modern 15 https://amzn.to/3PB

### Pretty-Print Tavily Results

Loop through the structured Tavily results and display title and content for each item.


In [58]:
for i, item in enumerate(results["results"], start=1):
    print(f"\nResult {i}")
    print("Title:", item["title"])
    print("Content:", item["content"])


Result 1
Title: Best laptop for programming with AI features under 1 lakhs?
Content: Best laptop for programming with AI features under 1 lakhs ... Laptop suggestions for programming beginners under Rs 27,000. Dakshta

Result 2
Title: The Ultimate AI/ML Laptop Buying Guide in 2026! - YouTube
Content: ... under $1000, affordable laptops for AI and deep learning ... best for AI development in 2024, best laptop for training AI models at home.

Result 3
Title: Best Laptops for AI Engineer (2026) Updated! - YouTube
Content: Best Buying Links 1. MSI Modern 15 https://amzn.to/3PB0eR9 ... best for AI development in 2024, best laptop for training AI models at home.

Result 4
Title: AI Laptops Price List In India (May 2026) | 91mobiles.com
Content: Top 5 AI Laptops with Prices & Specs ; MSI Prestige 13 AI Evo A1MG-051IN Laptop, 13.3" (33.78 cm), 2880 x 1800 px, OLED Display, Intel Core Ultra 7 155H, 4.8 GHz

Result 5
Title: Suggest a best laptop for Rs. 1-1.5 lakhs for AI Product Designer
Conte

### Define a Tavily Discovery Schema

This Pydantic model is designed for Tavily results. It captures brand, model, specs, source URL, and reason for each discovered laptop.


In [61]:
# ---- Understanding: Without vs With Pydantic ----
# Without pydantic: LLM returns random formats every time (plain text, random JSON, etc.)
# With pydantic: LLM is forced to return data in the exact schema we define
# This makes the output reliable and allows the next agent to consume it

from pydantic import BaseModel, Field
from typing import List


class Product(BaseModel):
    """Schema for a single discovered product (used by Tavily-based discovery)."""
    brand: str = Field(description="Product manufacturer")
    model: str = Field(description="Product model")
    specs: str = Field(description="Key specifications of the product")
    source: str = Field(description="URL or source of the product information")
    reason: str = Field(description="Why this product was discovered")


class DiscoveryOutput(BaseModel):
    """Complete discovery output — list of products found."""
    products: List[Product]

### Convert Tavily Results to Search Content

Build a single string (`search_content`) containing titles, URLs, and content from Tavily results. This string becomes the prompt context for the structured LLM.


In [62]:
search_content = ""

for item in results["results"]:
    search_content += f"""
Title: {item['title']}
URL: {item['url']}
Content: {item['content']}
Source: {item['url']}

"""

# structured output

structured_llm = llm.with_structured_output(
    DiscoveryOutput
)

# Prompt

# prompt = f"""
# You are a product discovery expert.

# Extract only valid laptop products.

# Search Results:

# {search_content}

# Return discovered products.
# """

prompt = f"""
You are a product discovery expert.

Analyze all search results carefully.

Extract ALL laptop products explicitly mentioned.

Requirements:
- Return every laptop model mentioned.
- Do not limit to one product.
- Extract up to 10 products.
- Ignore generic articles.
- Only include actual laptop models.

Search Results:

{search_content}
"""

# Agent Invoke

discovered_products = structured_llm.invoke(
    prompt
)

### Extract Structured Products from Tavily Context

Invoke `structured_llm` with a prompt that includes the search content. The LLM returns a `DiscoveryOutput` object containing all discovered products.


In [63]:
print(discovered_products)

products=[Product(brand='MSI', model='Modern 15', specs='General specifications for programming', source='https://www.youtube.com/watch?v=RE5NE-mBULk', reason='Recommended for AI development'), Product(brand='MSI', model='Prestige 13 AI Evo A1MG-051IN', specs='13.3", 2880 x 1800 px, OLED, Intel Core Ultra 7 155H', source='https://www.91mobiles.com/list-of-laptops/ai-laptops', reason='Featured in the top AI laptops list'), Product(brand='Alienware', model='m16 R1', specs='i7 13th gen, 1TB SSD, 32GB RAM, RTX 4050', source='https://www.reddit.com/r/laptops/comments/1tkq7l7/suggest_a_best_laptop_for_rs_115_lakhs_for_ai', reason='Personal recommendation for an AI Product Designer')]


### Print the Structured Tavily Output

Display the discovered products object to confirm the schema was followed.


In [64]:
print(discovered_products.model_dump())

{'products': [{'brand': 'MSI', 'model': 'Modern 15', 'specs': 'General specifications for programming', 'source': 'https://www.youtube.com/watch?v=RE5NE-mBULk', 'reason': 'Recommended for AI development'}, {'brand': 'MSI', 'model': 'Prestige 13 AI Evo A1MG-051IN', 'specs': '13.3", 2880 x 1800 px, OLED, Intel Core Ultra 7 155H', 'source': 'https://www.91mobiles.com/list-of-laptops/ai-laptops', 'reason': 'Featured in the top AI laptops list'}, {'brand': 'Alienware', 'model': 'm16 R1', 'specs': 'i7 13th gen, 1TB SSD, 32GB RAM, RTX 4050', 'source': 'https://www.reddit.com/r/laptops/comments/1tkq7l7/suggest_a_best_laptop_for_rs_115_lakhs_for_ai', 'reason': 'Personal recommendation for an AI Product Designer'}]}


### Convert to Dictionary

`discovered_products.model_dump()` shows the JSON-friendly representation, useful for logging or saving results.


In [65]:
for product in discovered_products.products:
    print("Brand :", product.brand)
    print("Model :", product.model)
    print("Reason:", product.reason)
    print("-" * 50)

Brand : MSI
Model : Modern 15
Reason: Recommended for AI development
--------------------------------------------------
Brand : MSI
Model : Prestige 13 AI Evo A1MG-051IN
Reason: Featured in the top AI laptops list
--------------------------------------------------
Brand : Alienware
Model : m16 R1
Reason: Personal recommendation for an AI Product Designer
--------------------------------------------------


### Iterate Over Discovered Products

Print brand, model, and reason for each product in a clean list. This confirms the extraction worked and the data is ready for downstream agents.
